# Project 2 · Experiment 1 — RoBERTa IID Evaluation
**独立可运行的 Jupyter Notebook / Windows + Anaconda + NVIDIA CUDA**

目的：使用固定 RAID `iid_train.csv` / `iid_validation.csv` / `iid_test.csv`，微调 **RoBERTa-base** 完成人类文本 vs AI 文本分类。

- **这不是 LOGO**：四种 AI 生成器都在训练集出现；测试的是未见过的新样本。
- **防止数据泄漏**：先核查 `source_id`、完全相同文本与行 ID 是否跨 split。
- **只用 Validation 选最佳模型**，完成后才进行一次最终 Test 评估。
- 保存总体/分生成器结果和固定的评测样本清单，之后 LOGO 可以严格复用相同测试样本。
- **没有任何预填充成绩**：所有测试指标都来自你自己的实际训练。

### 文件结构
将 Notebook 放在仓库的 `notebooks/` 文件夹中，并在仓库根目录准备 `raid_experiment_data/`：

```text
raid-roberta-cross-generator-detection/
├── notebooks/
│   └── RAID_RoBERTa_IID_Jupyter.ipynb   ← 本文件
└── raid_experiment_data/
    ├── iid_train.csv
    ├── iid_validation.csv
    └── iid_test.csv
```
**运行方式**：Jupyter Notebook / JupyterLab → 打开本文件 → 从上到下依次执行单元格（Shift+Enter）。第一次下载模型需要联网。

## 0. 安装依赖（第一次才需要）

**请先确认 Notebook 右上角选中的 Kernel 是你的目标 Anaconda Python 环境。**

1. 如缺少 `torch`，或当前 `torch.cuda.is_available()` 为 `False`，请打开 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/)；按 **Windows → Pip → Python → CUDA** 选择兼容的安装命令，并安装到**当前 Jupyter Kernel 所属环境**。不建议先猜一个 CUDA 版本。
2. 如缺少 Transformers 或其他依赖，取消下一格 `%pip` 命令前面的注释并运行；安装后按 Jupyter 提示**重启 Kernel**，再从头执行。
3. 不需要单独运行此前的 `.py` 训练脚本；本 Notebook 是自包含版本。

**注意：** 不要在有重要训练进程运行时升级 PyTorch / Transformers。

In [ ]:
# 第一次使用时，如缺少相关依赖：删除下一行开头的 # 再运行。
# %pip install "transformers>=4.50,<5" "pandas>=2,<3" "numpy>=1.24,<3" "scikit-learn>=1.3,<2" "tqdm>=4.60" "safetensors>=0.4"

import sys
print("Notebook kernel Python:", sys.executable)
print("Python version:", sys.version.split()[0])


## 1. 参数与文件路径

使用仓库根目录下由数据处理脚本生成的 `raid_experiment_data/`。Notebook 放在 `notebooks/` 子文件夹即可，路径会从当前工作目录向上自动查找。如果缺少 CSV，请先从仓库根目录运行 `python scripts/fetch_raid_balanced_final.py` 和 `python scripts/prepare_raid_splits_v2.py`。

初始配置：RoBERTa-base、最多 3 轮、最大 256 token、batch 4、梯度累积 4、学习率 `2e-5`。低显存时可改成 batch 2 + 梯度累积 8。**更改参数会影响历史结果的严格复现；IID 与 LOGO 应保持相同参数。**


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Find repository root from Jupyter launched in the root or notebooks/."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "raid_experiment_data").is_dir():
            return candidate
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the repository root. Run Jupyter inside the cloned "
        "repository and generate raid_experiment_data/ using the scripts first."
    )

PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "raid_experiment_data"
OUTPUT_DIR = PROJECT_DIR / "iid_roberta_results_jupyter"

MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 2
SEED = 42
ALLOW_CPU = False  # 建议 NVIDIA 用户保留 False；避免意外开始极慢的 CPU 训练

print("Project root:", PROJECT_DIR.resolve())
print("Data folder: ", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
assert DATA_DIR.is_dir(), (
    f"找不到数据文件夹：{DATA_DIR}。请检查 PROJECT_DIR，"
    "请先从仓库根目录运行 scripts/ 中的数据准备脚本。"
)


## 2. GPU / CUDA 环境检查（不训练）

这里先验证当前 Jupyter **Kernel** 能真正调用 CUDA，而不仅仅是电脑装有 NVIDIA 显卡。必须看到 `CUDA usable: True` 才建议继续。

In [ ]:
import platform, subprocess
try:
    import torch
except ImportError as exc:
    raise RuntimeError("当前 Jupyter Kernel 缺少 PyTorch，请先安装 CUDA 版 PyTorch。") from exc

print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA usable:", torch.cuda.is_available())
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=12)
    if r.returncode == 0:
        print("nvidia-smi (first 1000 chars):\n", r.stdout[:1000])
    else:
        print("nvidia-smi status:", r.stderr[:300])
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("nvidia-smi not detected here; rely on the PyTorch CUDA test below.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.2f} GiB")
    probe = torch.tensor([1.0, 2.0], device="cuda")
    print("CUDA tensor test:", (probe * 2).cpu().tolist())
    print("[PASS] CUDA 在本 Notebook Kernel 中可用。")
else:
    print("[WARNING] CUDA 不可用；不要开始训练。请检查当前 Kernel 和 CUDA 版 PyTorch 安装。")
    if not ALLOW_CPU:
        print("下一步模型初始化将停止，避免意外使用 CPU 训练。")


## 3. 核对固定 IID 数据划分（不训练）

本阶段会直接读取三份 CSV、核对 0/1 标签、预期生成器、文件 SHA-256，以及各 split 间的 `source_id`、完全相同文本与原始行 `id`。**不会重新切分数据**。检查通过后才进入模型训练。

In [ ]:
import hashlib, json, math, random, gc
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)
from IPython.display import display

SPLIT_FILES = {"train": "iid_train.csv", "validation": "iid_validation.csv", "test": "iid_test.csv"}
REQUIRED = {"id", "source_id", "text", "label", "model", "domain"}
LABEL_NAMES = {0: "Human", 1: "AI"}
EXPECTED_AI_MODELS = {"gpt4", "chatgpt", "llama-chat", "mistral-chat"}

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def source_group(row):
    source = str(row["source_id"]).strip()
    if source:
        return "src::" + source
    return "missing_source::" + str(row["id"])

def read_and_validate(data_dir: Path):
    splits = {}
    file_hashes = {}
    for split, filename in SPLIT_FILES.items():
        path = data_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Missing {path}. Run prepare_raid_splits_v2.py first.")
        # Disable CSV default NA parsing: the literal text 'NA' is not missing.
        df = pd.read_csv(path, keep_default_na=False, encoding="utf-8-sig")
        missing = REQUIRED - set(df.columns)
        if missing:
            raise ValueError(f"{path}: missing required columns {sorted(missing)}")
        if df.empty:
            raise ValueError(f"{path}: empty dataset")
        for name in ["id", "text", "model", "domain"]:
            if df[name].astype(str).str.strip().eq("").any():
                raise ValueError(f"{path}: blank {name} values found")
        df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
        if not set(df["label"].unique()).issubset({0, 1}):
            raise ValueError(f"{path}: expected binary labels 0/1")
        # RAID model codes are used for dataset checks. Keep exact original metadata.
        is_human = df["model"].astype(str).str.lower().eq("human")
        if (is_human != df["label"].eq(0)).any():
            raise ValueError(f"{path}: inconsistent 'human' model vs label=0")
        if df["id"].duplicated().any():
            raise ValueError(f"{path}: repeated row ids found")
        df["_group"] = df.apply(source_group, axis=1)
        df["_text_hash"] = df["text"].map(lambda x: hashlib.sha256(str(x).encode("utf-8")).hexdigest())
        splits[split] = df
        file_hashes[filename] = file_sha256(path)

    for i, a in enumerate(SPLIT_FILES):
        for b in list(SPLIT_FILES)[i+1:]:
            left, right = splits[a], splits[b]
            n_groups = len(set(left["_group"]) & set(right["_group"]))
            n_texts = len(set(left["_text_hash"]) & set(right["_text_hash"]))
            n_ids = len(set(left["id"]) & set(right["id"]))
            if n_groups or n_texts or n_ids:
                raise ValueError(
                    f"Leakage {a} vs {b}: groups={n_groups}, "
                    f"identical_texts={n_texts}, duplicated_ids={n_ids}"
                )
    # The IID evaluation needs every intended generator in the training data.
    train_ai_models = set(splits["train"].loc[splits["train"]["label"] == 1, "model"].astype(str))
    if train_ai_models != EXPECTED_AI_MODELS:
        raise ValueError(
            f"AI models in train: {sorted(train_ai_models)}; "
            f"expected {sorted(EXPECTED_AI_MODELS)}. Check dataset/model codes."
        )
    for split, df in splits.items():
        if set(df["label"].unique()) != {0, 1}:
            raise ValueError(f"{split} must contain both Human and AI")
        if split != "train":
            models = set(df.loc[df["label"] == 1, "model"].astype(str))
            if models != EXPECTED_AI_MODELS:
                raise ValueError(f"{split} missing expected AI generators: {EXPECTED_AI_MODELS - models}")

    report = {
        "datasets": {
            key: {
                "n": len(df),
                "human": int((df["label"] == 0).sum()),
                "ai": int((df["label"] == 1).sum()),
                "source_groups": int(df["_group"].nunique()),
                "model_counts": {str(k): int(v) for k, v in df["model"].value_counts().items()},
                "domain_counts": {str(k): int(v) for k, v in df["domain"].value_counts().items()},
            } for key, df in splits.items()
        },
        "verified": "zero source-group, exact-text and row-id cross-split overlap",
        "file_sha256": file_hashes,
    }
    return splits, report

def make_generator_eval_manifest(test: pd.DataFrame, seed: int):
    """For each target generator, create a domain-matched Human+AI TEST set.

    The AI target and Human controls come exclusively from the immutable IID test
    partition. Reuse this CSV *unchanged* for corresponding LOGO evaluations.
    """
    human = test[test["label"] == 0]
    ai = test[test["label"] == 1]
    parts = []
    rng = np.random.default_rng(seed)
    for model in sorted(ai["model"].unique()):
        target = ai[ai["model"] == model]
        human_pieces = []
        for domain, target_count in target["domain"].value_counts().sort_index().items():
            pool = human[human["domain"] == domain]
            if len(pool) < target_count:
                raise ValueError(
                    f"Not enough Human TEST samples in {domain} "
                    f"for {model} ({len(pool)} < {target_count})"
                )
            chosen_idx = rng.choice(pool.index.to_numpy(), size=int(target_count), replace=False)
            human_pieces.append(human.loc[chosen_idx])
        matched = pd.concat([target] + human_pieces, ignore_index=True)
        if len(matched) != 2 * len(target):
            raise AssertionError("Domain-match construction failed")
        # Label only which generator is being evaluated; each Human sample can
        # appear in multiple separate target-generator evaluations (intentional).
        matched = matched[["id", "source_id", "model", "domain", "label"]].copy()
        matched.insert(0, "target_generator", str(model))
        parts.append(matched)
    return pd.concat(parts, ignore_index=True)

def scores(y_true, y_pred, ai_proba):
    yt, yp = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    result = {
        "n": len(yt),
        "accuracy": float(accuracy_score(yt, yp)),
        "precision_ai": float(precision_score(yt, yp, zero_division=0)),
        "recall_ai": float(recall_score(yt, yp, zero_division=0)),
        "f1_ai": float(f1_score(yt, yp, zero_division=0)),
        "f1_macro": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "confusion_matrix_human_ai": confusion_matrix(yt, yp, labels=[0, 1]).tolist(),
    }
    if len(set(yt)) == 2:
        result["roc_auc"] = float(roc_auc_score(yt, ai_proba))
    else:
        result["roc_auc"] = None
    return result

def save_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def set_seeds(seed, torch):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
splits, data_report = read_and_validate(DATA_DIR)
save_json(OUTPUT_DIR / "data_preflight_report.json", data_report)
summary = pd.DataFrame([
    {"split": name, **{k: info[k] for k in ["n", "human", "ai", "source_groups"]}}
    for name, info in data_report["datasets"].items()
])
display(summary)
print("[PASS] 三个 split 无来源组 / 完全相同文本 / 行 ID 交叉重叠。")
print("[SAVED]", OUTPUT_DIR / "data_preflight_report.json")


## 4. 初始化 RoBERTa 和数据加载器

这一格第一次运行需要联网下载 `FacebookAI/roberta-base` tokenizer 和权重。**请确认 CUDA / 数据检查通过后再执行**。如果显存不足，先在第 1 步降低 batch size，**重启 Kernel** 后从头运行，不要在半次训练中直接修改模型参数。

In [ ]:
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("CUDA 不可用，训练已停止。请按第 0-2 步修复 Kernel / PyTorch 安装。")

try:
    import transformers
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer,
        DataCollatorWithPadding, get_linear_schedule_with_warmup
    )
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm
except ImportError as exc:
    raise RuntimeError("缺少 transformers/tqdm 依赖，请返回第 0 步安装后重启 Kernel。") from exc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
set_seeds(SEED, torch)

RUN_CONFIG = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO, "patience": PATIENCE, "seed": SEED,
    "data_dir": str(DATA_DIR.resolve()), "output_dir": str(OUTPUT_DIR.resolve()),
    "torch_version": torch.__version__, "transformers_version": transformers.__version__,
    "python_version": platform.python_version(), "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if USE_AMP else None,
    "data_sha256": data_report["file_sha256"],
    "checkpoint_selection": "validation_f1_ai",
    "test_rule": "test evaluated only after selecting best validation checkpoint"
}
save_json(OUTPUT_DIR / "run_config.json", RUN_CONFIG)
print("[MODEL] Loading", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.model_max_length >= MAX_LENGTH, "MAX_LENGTH 超过该 tokenizer 的允许长度"

class TokenizedTextDataset(Dataset):
    def __init__(self, df):
        self.tokens = tokenizer(
            df["text"].astype(str).tolist(),
            truncation=True, max_length=MAX_LENGTH, padding=False,
            return_attention_mask=True
        )
        self.labels = df["label"].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.tokens["input_ids"][idx],
            "attention_mask": self.tokens["attention_mask"][idx],
            "labels": self.labels[idx]
        }

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8 if USE_AMP else None,
    return_tensors="pt"
)
train_set = TokenizedTextDataset(splits["train"])
val_set = TokenizedTextDataset(splits["validation"])
loader_rng = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collator, generator=loader_rng)
val_loader = DataLoader(val_set, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=LABEL_NAMES,
    label2id={"Human": 0, "AI": 1}
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_updates = updates_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_updates * WARMUP_RATIO),
    num_training_steps=total_updates
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"[READY] device={DEVICE} | train={len(train_set)} | validation={len(val_set)}")
print("提示：随机初始化 classification head 的警告属于正常现象。")


## 5. 定义评估函数
使用 `0.5` 的标准分类阈值（`argmax`）。后续只根据验证集选择 epoch；不会根据测试集调整阈值。

In [ ]:
def evaluate_loader(model, loader, description):
    model.eval()
    truth, preds, proba, loss_values = [], [], [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=description, leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(**batch)
            prob = torch.softmax(out.logits.float(), dim=-1)[:, 1]
            truth.extend(batch["labels"].cpu().tolist())
            preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
            proba.extend(prob.cpu().tolist())
            loss_values.append(float(out.loss.item()))
    result = scores(truth, preds, proba)
    result["mean_eval_loss"] = float(np.mean(loss_values))
    return result, preds, proba

print("[READY] evaluation function defined; no training/test evaluation has run yet.")


## 6. 开始训练（本 Notebook 最耗时的单元格）

这格会运行至多 3 个 epoch，以 `validation_f1_ai` 保存最佳模型。正式训练可能需要较长时间，具体取决于显卡与网络。**不要中断 Kernel、重复运行本单元格或在训练中同时修改其他配置**。

如果报 `CUDA out of memory`：重启 Kernel、返回第 1 步，将 `BATCH_SIZE=2`、`EVAL_BATCH_SIZE=2`、`GRAD_ACCUM=8`，必要时 `MAX_LENGTH=192`；然后从头执行。变更配置要在结果中记录，之后 LOGO 保持一致。

In [ ]:
checkpoint_dir = OUTPUT_DIR / "best_model"
best_val_f1 = -1.0
best_epoch = -1
non_improve = 0
history = []

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_train_loss = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"), start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            group_start = ((step - 1) // GRAD_ACCUM) * GRAD_ACCUM
            group_len = min(GRAD_ACCUM, len(train_loader) - group_start)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(**batch)
                loss = out.loss / group_len
            scaler.scale(loss).backward()
            total_train_loss += float(out.loss.item())

            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_metrics, _, _ = evaluate_loader(model, val_loader, f"Validation epoch {epoch}")
        current_val_f1 = val_metrics["f1_ai"]
        record = {
            "epoch": epoch,
            "mean_train_loss": total_train_loss / len(train_loader),
            "validation_accuracy": val_metrics["accuracy"],
            "validation_f1_ai": current_val_f1,
            "validation_f1_macro": val_metrics["f1_macro"],
            "validation_recall_ai": val_metrics["recall_ai"],
        }
        history.append(record)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
        print(f"Epoch {epoch}: train_loss={record['mean_train_loss']:.4f}; "
              f"validation_f1_ai={current_val_f1:.4f}; "
              f"validation_accuracy={record['validation_accuracy']:.4f}")

        if current_val_f1 > best_val_f1 + 1e-8:
            best_val_f1 = current_val_f1
            best_epoch = epoch
            non_improve = 0
            model.save_pretrained(checkpoint_dir, safe_serialization=True)
            tokenizer.save_pretrained(checkpoint_dir)
            print(f"[BEST] saved epoch {best_epoch} at {checkpoint_dir}")
        else:
            non_improve += 1
            if non_improve >= PATIENCE:
                print("[EARLY STOP] no validation F1 improvement.")
                break
except torch.cuda.OutOfMemoryError as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "GPU 显存不足。请重启 Kernel 并在第1步设置 "
        "BATCH_SIZE=2, EVAL_BATCH_SIZE=2, GRAD_ACCUM=8；必要时 MAX_LENGTH=192。"
    ) from exc

assert best_epoch > 0, "没有成功保存任何验证最佳 checkpoint"
print(f"[TRAIN DONE] best_epoch={best_epoch}; best_val_f1={best_val_f1:.4f}")
display(pd.DataFrame(history))


## 7. 最终 IID Test（仅在训练结束后运行一次）

**重要：** 这是最终测试，不用于挑选模型或调参。测试集在本实验训练期间始终保持固定。

输出包含总体分数、逐样本预测结果、每个生成器配对领域的 Human 对照集及其评估分数。保存的 `per_generator_test_manifest.csv` 是未来 LOGO 对照不可替换的测试名单。

In [ ]:
if "best_epoch" not in globals() or best_epoch < 1 or not checkpoint_dir.is_dir():
    raise RuntimeError("尚无成功训练的最佳模型；请先完成第 6 步。")

# 释放训练模型和优化器的显存，然后读取 Validation 选择的 checkpoint。
del model, optimizer, scheduler, scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(DEVICE)
model.eval()

test_set = TokenizedTextDataset(splits["test"])
test_loader = DataLoader(test_set, batch_size=EVAL_BATCH_SIZE,
                         shuffle=False, num_workers=0, collate_fn=collator)
test_metrics, test_preds, test_proba = evaluate_loader(model, test_loader, "FINAL IID TEST")
test_metrics.update({
    "selected_epoch": best_epoch,
    "best_validation_f1": best_val_f1,
    "test_data_count": len(splits["test"])
})
save_json(OUTPUT_DIR / "iid_test_metrics.json", test_metrics)

test_df = splits["test"].drop(columns=["_group", "_text_hash"]).copy()
test_df["prediction"] = test_preds
test_df["probability_ai"] = test_proba
test_df["correct"] = test_df["prediction"].eq(test_df["label"])
export_cols = [c for c in ["id", "source_id", "model", "generator", "domain", "label",
                           "prediction", "probability_ai", "correct"] if c in test_df.columns]
test_df[export_cols].to_csv(OUTPUT_DIR / "iid_test_predictions.csv", index=False)

manifest = make_generator_eval_manifest(splits["test"], SEED)
manifest.to_csv(OUTPUT_DIR / "per_generator_test_manifest.csv", index=False)
by_id = test_df.set_index("id")
rows = []
for target_gen, part in manifest.groupby("target_generator", sort=True):
    matched = by_id.loc[part["id"].tolist()]
    metrics = scores(matched["label"], matched["prediction"], matched["probability_ai"])
    metrics.update({
        "target_generator": target_gen,
        "n_human": int((matched["label"] == 0).sum()),
        "n_ai": int((matched["label"] == 1).sum())
    })
    metrics.pop("confusion_matrix_human_ai")
    rows.append(metrics)
per_generator = pd.DataFrame(rows)
per_generator.to_csv(OUTPUT_DIR / "iid_per_generator_metrics.csv", index=False)

print("[FINISHED] Final IID TEST summary:")
display(pd.DataFrame([test_metrics]).drop(columns=["confusion_matrix_human_ai"]))
print("Confusion matrix (rows: true Human, AI; columns: predicted Human, AI):")
print(np.asarray(test_metrics["confusion_matrix_human_ai"]))
print("Per-generator matched evaluation:")
display(per_generator[["target_generator", "n_human", "n_ai", "accuracy", "precision_ai", "recall_ai", "f1_ai", "f1_macro"]])
print("[RESULTS SAVED]", OUTPUT_DIR.resolve())
print("请勿根据 Test 结果返回重调超参数；后续 LOGO 必须复用 per_generator_test_manifest.csv。")


## 8. 需要保存和发回来的文件

成功运行后，`iid_roberta_results_jupyter` 文件夹中会有：

| 文件 | 含义 |
|---|---|
| `data_preflight_report.json` | 防泄漏检查及输入 CSV 的 SHA-256 |
| `run_config.json` | 使用的模型、训练参数、CUDA 环境信息 |
| `training_history.csv` | 每个 epoch 的 Train loss / Validation 分数 |
| `best_model/` | Validation F1 最佳的模型权重和 tokenizer |
| `iid_test_metrics.json` | 最终 IID 测试总体指标 |
| `iid_test_predictions.csv` | 逐条测试预测（不附完整原文） |
| `iid_per_generator_metrics.csv` | 四种生成器分别计算的二分类指标 |
| `per_generator_test_manifest.csv` | **后续 LOGO 必须复用的测试样本名单** |

实验结束后，请将 `iid_test_metrics.json`、`iid_per_generator_metrics.csv`、`training_history.csv` 和 `run_config.json` 上传给我。我们再设计 **Experiment 2: LOGO**。

**研究局限**：`MAX_LENGTH=256` 会截断较长文本；文本长度/格式可能是混杂因素；LOGO 排除一种生成器会改变训练数据构成。Poster 中要如实报告，不能仅靠 F1 差异断言模型“真正理解”AI 文本。